In [2]:
import os
import yfinance as yf
from sec_edgar_downloader import Downloader
import requests
import pandas as pd
from newsapi import NewsApiClient

# 환경변수에서 API 키 읽기
ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
NEWSAPI_API_KEY        = os.getenv("NEWSAPI_API_KEY")

tickers = ["AAPL","AMZN","NVDA","MSFT","GOOGL"]

In [3]:
for t in tickers:
    ticker = yf.Ticker(t)
    hist = ticker.history(period="1y")  # 과거 1년 일별 시세
    print(f"{t} - 최근 종가:\n", hist["Close"].tail(3), "\n")

AAPL - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    213.880005
2025-07-28 00:00:00-04:00    214.050003
2025-07-29 00:00:00-04:00    212.580002
Name: Close, dtype: float64 

AMZN - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    231.440002
2025-07-28 00:00:00-04:00    232.789993
2025-07-29 00:00:00-04:00    230.980103
Name: Close, dtype: float64 

NVDA - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    173.500000
2025-07-28 00:00:00-04:00    176.750000
2025-07-29 00:00:00-04:00    177.235001
Name: Close, dtype: float64 

MSFT - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    513.710022
2025-07-28 00:00:00-04:00    512.500000
2025-07-29 00:00:00-04:00    513.109985
Name: Close, dtype: float64 

GOOGL - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    193.179993
2025-07-28 00:00:00-04:00    192.580002
2025-07-29 00:00:00-04:00    193.399597
Name: Close, dtype: float64 



In [ ]:
from sec_edgar_downloader import Downloader

dl = Downloader(
    "GEN",                
    "james4327@gmail.com",            
    download_folder="sec"     
)

for t in tickers:
    dl.get("10-K", t, limit=1)

print("10-K 다운로드 완료 → notebook/ 폴더 확인")

10-K 다운로드 완료 → notebook/ 폴더 확인


In [5]:
def fetch_av_data(symbol, function):
    url = "https://www.alphavantage.co/query"
    params = {
        "function": function,
        "symbol":   symbol,
        "apikey":   ALPHA_VANTAGE_API_KEY
    }
    r = requests.get(url, params=params)
    data = r.json()
    # key 이름이 다르므로 함수별로 꺼내기
    key_map = {
      "INCOME_STATEMENT": "quarterlyReports",
      "BALANCE_SHEET":    "quarterlyReports",
      "CASH_FLOW":        "quarterlyReports"
    }
    return pd.DataFrame(data.get(key_map[function], []))

# 예시: AAPL 분기별 손익계산서
df_income = fetch_av_data("AAPL", "INCOME_STATEMENT")
print("AAPL 최근 3개 분기 손익:\n", df_income.head(3))

AAPL 최근 3개 분기 손익:
 Empty DataFrame
Columns: []
Index: []


In [6]:
newsapi = NewsApiClient(api_key=NEWSAPI_API_KEY)
all_news = {}
for t in tickers:
    articles = newsapi.get_everything(q=t,
                                      language='en',
                                      sort_by='publishedAt',
                                      page_size=20)['articles']
    df = pd.DataFrame(articles)[["publishedAt","source","title","url"]]
    all_news[t] = df
    print(f"{t} 뉴스 샘플:\n", df.head(3), "\n")

TypeError: expected string or bytes-like object, got 'NoneType'